# PCA for Embedding Vectors

### 1. Imports and configurations

In [ ]:
import os, re, json
import pickle
import numpy as np
import pandas as pd
from pathlib import Path
from langchain_ollama import OllamaEmbeddings
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from matplotlib.lines import Line2D

# Configurations
# Add the directory path for the case files
BASE_DIR = (
    "cases" 
)
# Add the directory path for where you want to save the results
OUT_DIR = (
    "PCA_embeddinggemma"
)
# Write the model name how it is installed on your device
OLLAMA_MODEL = "embeddinggemma:F32"
# Default Ollama address for your own device
OLLAMA_URL = "http://127.0.0.1:11434"

os.makedirs(OUT_DIR, exist_ok=True)

### 2. Data source import

The source data looks like this (syntethic original cases with edits in 3 pertubations):
<pre>
cases
├─ english_cases
│  ├─ medicine_1
│  │  ├─ original
│  │  │  └─ Case_1_Medicine_English_summary.csv
│  │  ├─ delete
│  │  │  ├─ Case_1_Medicine_English_summary_delete_1.csv
│  │  │  ├─ Case_1_Medicine_English_summary_delete_2.csv
│  │  │  └─ Case_1_Medicine_English_summary_delete_3.csv
│  │  ├─ modify
│  │  │  ├─ Case_1_Medicine_English_summary_modify_1.csv
│  │  │  ├─ Case_1_Medicine_English_summary_modify_2.csv
│  │  │  └─ Case_1_Medicine_English_summary_modify_3.csv
│  │  └─ paraphrase
│  │     ├─ Case_1_Medicine_English_summary_paraphrase_1.csv
│  │     ├─ Case_1_Medicine_English_summary_paraphrase_2.csv
│  │     └─ Case_1_Medicine_English_summary_paraphrase_3.csv
│  ├─ medicine_2 … same structure …
│  ├─ medicine_3
│  ├─ medicine_4
│  └─ medicine_5
├─ finnish_cases
│  └─ medicine_1..medicine_5 (same structure)
└─ swedish_cases
   └─ medicine_1..medicine_5 (same structure)
</pre>

In [11]:
# Walk the dataset folders and collect every document's text and metadata

def read_text(filepath: str) -> str: # Read CSV file as plain text and strip trailing whitespace
    with open(filepath, "r", encoding="utf-8") as f:
        return f.read().strip()


def parse_severity(name: str) -> str: # Severity level from filename
    m = re.search(r"(?:^|[_-])(1|2|3)(?=\.|$)", name)
    return m.group(1) if m else ""


records = []  # Empty list to hold one dict per document

# The main loop, that loops through the three language folders
for lang_folder in ["english_cases", "finnish_cases", "swedish_cases"]:
    lang = lang_folder.replace("_cases", "")  # "english_cases" -> "english", etc., store as the language label
    lang_path = os.path.join(BASE_DIR, lang_folder) # Path to the language folder

    for case_id in sorted(os.listdir(lang_path)):  # Inside each language folder, loop over each case folder (1-5)
        case_path = os.path.join(lang_path, case_id) # Path to the case folder

        for typ in ["original", "delete", "modify", "paraphrase"]:  # Within each case, loop thorugh the original + 3 variation folders
            pdir = os.path.join(case_path, typ) # Path to the directory (variation folder)

            for filename in sorted(os.listdir(pdir)):  # For each variation folder, read all .csv documents
                if not filename.endswith(".csv"): # Only .csv files
                    continue
                filepath = os.path.join(pdir, filename)
                text = read_text(filepath)

                # Append a record with both metadata and the raw text
                records.append(
                    {
                        "doc_id": f"{lang}:{case_id}:{typ}:{filename}",  # Unique id
                        "language": lang,  # english / finnish / swedish
                        "case_id": case_id,  # Case folder name
                        "type": typ,  # original / delete / modify / paraphrase
                        "severity": parse_severity(filename),  # "1" / "2" / "3" or ""
                        "file": filename,  
                        "text": text, 
                        "n_chars": len(text),  # Simple length metric
                    }
                )

# Convert to a DataFrame for easier downstream work
meta = pd.DataFrame(records)
print("Docs found:", len(meta))
meta.head(4)


Docs found: 150


,doc_id,language,case_id,type,severity,file,text,n_chars
0,english:medicine_1:original:Case_1_Medicine_En...,english,medicine_1,original,,Case_1_Medicine_English_summary.csv,"""## Reason of admission\n69 years old man who ...",1487
1,english:medicine_1:delete:Case_1_Medicine_Engl...,english,medicine_1,delete,1,Case_1_Medicine_English_summary_delete_1.csv,"""## Reason of admission\n69 years old man who ...",1367
2,english:medicine_1:delete:Case_1_Medicine_Engl...,english,medicine_1,delete,2,Case_1_Medicine_English_summary_delete_2.csv,"""## Reason of admission\n69 years old man who ...",1231
3,english:medicine_1:delete:Case_1_Medicine_Engl...,english,medicine_1,delete,3,Case_1_Medicine_English_summary_delete_3.csv,"""## Reason of admission\n69 years old man who ...",1054


### 3. Embedding Vectors
Data processing, to get the embedding vectors that are used as the input data for the PCA.

In [12]:
# Really simple build as it is not the main focus of the course, no batching, etc.
embedder = OllamaEmbeddings(model=OLLAMA_MODEL, base_url=OLLAMA_URL) # Initialize the Ollama embedder

texts = meta["text"].tolist() # List of all texts to embed

emb_list = embedder.embed_documents(texts)  # From Ollama for embeddings for all texts at once, output is python list

X = np.asarray(emb_list, dtype=np.float32)  # Convert to a NumPy matrix X of shape (N, d)

# Save embeddings and metadata to files
N, d = X.shape
np.savez(os.path.join(OUT_DIR, "embedding_vectors.npz"), X=X)
meta.drop(columns=["text"]).to_csv(os.path.join(OUT_DIR, "metadata.csv"), index=False)

print(f"Built embeddings: X shape = {X.shape}  (N docs x d dims)")
print(f"Embeddings: N={N}, d={d}")

meta.head(4) # Show the first 4 records

Built embeddings: X shape = (150, 4096)  (N docs x d dims)
Embeddings: N=150, d=4096


,doc_id,language,case_id,type,severity,file,text,n_chars
0,english:medicine_1:original:Case_1_Medicine_En...,english,medicine_1,original,,Case_1_Medicine_English_summary.csv,"""## Reason of admission\n69 years old man who ...",1487
1,english:medicine_1:delete:Case_1_Medicine_Engl...,english,medicine_1,delete,1,Case_1_Medicine_English_summary_delete_1.csv,"""## Reason of admission\n69 years old man who ...",1367
2,english:medicine_1:delete:Case_1_Medicine_Engl...,english,medicine_1,delete,2,Case_1_Medicine_English_summary_delete_2.csv,"""## Reason of admission\n69 years old man who ...",1231
3,english:medicine_1:delete:Case_1_Medicine_Engl...,english,medicine_1,delete,3,Case_1_Medicine_English_summary_delete_3.csv,"""## Reason of admission\n69 years old man who ...",1054


#### 3.1. Data characteristics

In [13]:
N, d = X.shape  # Use the in-memory X and meta from previous cell
print(f"Embeddings: N={N}, d={d}") # Check that I have the expected number of documents and dimensions

# Build a small summary table of document lengths by language
len_stats = (
    meta.groupby("language")["n_chars"]
    .agg(n="count", mean="mean", sd="std", min="min", max="max")
    .reset_index()
)
print("Document length stats by language:")
display(len_stats)  # Jupyter helper to show the table nicely

by_type_lang = meta.groupby(["language", "type"]).size().rename("count").reset_index()  # Balanced counts (optional, helpful check)
print("Document counts by language and type:")
display(by_type_lang.head(12))

# Save tables as csv files
summary_dir = os.path.join(OUT_DIR, "summary")
os.makedirs(summary_dir, exist_ok=True)
len_stats.to_csv(os.path.join(summary_dir, "length_stats_by_language.csv"), index=False)
by_type_lang.to_csv(
    os.path.join(summary_dir, "counts_by_language_type.csv"), index=False
)


Embeddings: N=150, d=4096
Document length stats by language:


,language,n,mean,sd,min,max
0,english,50,2069.22,469.878851,1054,2913
1,finnish,50,2100.90,477.406952,1070,2935
2,swedish,50,1913.30,430.733368,936,2645


Document counts by language and type:


,language,type,count
0,english,delete,15
1,english,modify,15
2,english,original,5
3,english,paraphrase,15
4,finnish,delete,15
5,finnish,modify,15
6,finnish,original,5
7,finnish,paraphrase,15
8,swedish,delete,15
9,swedish,modify,15


#### 3.2. Mean-centering the data for the PCA

μ (“mu”) is the mean embedding vector across all documents. \
Xc is the centered data matrix.

In [14]:
# Preparing for PCA: centering and variance per dimension
mu = X.mean(axis=0, keepdims=True)  # Compute mean vector mu across rows (documents) for each embedding dimension, keepdims=True keeps mu as shape (1, d)
Xc = X - mu  # Subtract the mean vector from every row so the centered matrix has column-wise mean ~0

var = Xc.var(axis=0, ddof=1)  # Per-dimension (i.e., per embedding coordinate) variance to spot outlier dimensions
rank_idx = np.argsort(var)[::-1] # Indices that sorts the variance array in descending order

# Save PCA input and variance table 
np.savez(os.path.join(OUT_DIR, "pca_input_centered.npz"), Xc=Xc, mu=mu.squeeze())
pd.DataFrame({"dim": np.arange(d), "variance": var}).to_csv(
    os.path.join(OUT_DIR, "variance_per_dimension.csv"), index=False, sep=";"
)

# Console summary for the report
print(f"Centered matrix shape: {Xc.shape}  (N docs x d dims)") # Sanity check, should be same shape as X
print("Mean of centered matrix (first 5 dims):", Xc.mean(axis=0)[:5]) # Should be very close to 0
print("Top 10 highest-variance dimensions:")  # Top 10 highest-variance dimensions. These are candidates for "outlier" coordinates
for k in range(10):
    j = int(rank_idx[k])
    print(f"  rank {k + 1:2d}: dim {j:4d}, variance {var[j]:.6f}")


Centered matrix shape: (150, 4096)  (N docs x d dims)
Mean of centered matrix (first 5 dims): [-2.2351741e-10  9.4374020e-10  2.9802323e-09 -3.3775966e-09
  2.5145710e-09]
Top 10 highest-variance dimensions:
  rank  1: dim 2109, variance 0.000801
  rank  2: dim 1443, variance 0.000627
  rank  3: dim  702, variance 0.000559
  rank  4: dim 2652, variance 0.000552
  rank  5: dim 2766, variance 0.000542
  rank  6: dim  374, variance 0.000515
  rank  7: dim  787, variance 0.000487
  rank  8: dim 3087, variance 0.000468
  rank  9: dim 2713, variance 0.000439
  rank 10: dim 1092, variance 0.000434


### 4. Principal component analysis (PCA)

In [15]:
OUT = os.path.join(OUT_DIR, "pca_simple") # Output folder for PCA results
os.makedirs(OUT, exist_ok=True)

# Fit PCA on centered data
N, d = Xc.shape  # N = number of documents, d = embedding dimensions
k = min(N - 1, d)  # Maximum useful number of PCs is rank ≤ min(N-1, d)
pca_doc = PCA(n_components=k, svd_solver="randomized", random_state=42) # PCA, using randomized SVD for speed and random state for reproducibility
Z = pca_doc.fit_transform(Xc)  # Fit PCA and get scores Z of shape (N, k)

evr = pca_doc.explained_variance_ratio_ # Explained variance ratio for each PC
cev = np.cumsum(evr) # Cumulative explained variance
p95 = int(np.searchsorted(cev, 0.95) + 1) # Number of PCs needed to reach 95% variance
print(f"PCs to reach 95% variance: {p95}")
print("First 10 EV ratios:", np.round(evr[:10], 4))
print("Sum of the EV ratios (or similar): " + str(sum(evr)))

PCs to reach 95% variance: 28
First 10 EV ratios: [0.1969 0.1372 0.1172 0.1058 0.0973 0.0739 0.0269 0.0205 0.0192 0.0175]
Sum of the EV ratios (or similar): 0.9999998


In [16]:
from matplotlib.lines import Line2D

# PLOTS
# Scree
plt.figure(figsize=(7, 4))  # New figure for the scree plot
plt.plot(
    np.arange(1, k + 1), evr, marker="o", linewidth=1
)  # EVR = explained variance ratio for each PC
plt.xlabel("Principal component")
plt.ylabel("Explained variance ratio")
plt.title("Scree plot")
plt.tight_layout()
plt.savefig(os.path.join(OUT, "scree.png"), dpi=200)  
plt.close()

# Cumulative explained variance
plt.figure(figsize=(7, 4))  
plt.plot(
    np.arange(1, k + 1), cev, marker="o", linewidth=1
)  # CEV = cumulative explained variance
plt.axhline(0.95, linestyle="--")  # Visual 95 percent threshold
plt.xlabel("Principal component")
plt.ylabel("Cumulative explained variance")
plt.title(f"Cumulative EV (95% at PC {p95})")
plt.tight_layout()
plt.savefig(os.path.join(OUT, "cumulative_ev.png"), dpi=200)  
plt.close()


# Anisotropy curve: sorted per-dimension variance, with Gini and participation ratio
def _gini_nonneg(x):
    x = np.asarray(x, float)
    x = x[x >= 0]  # Keep nonnegative values only
    if x.size == 0 or np.isclose(x.sum(), 0):
        return 0.0
    xs = np.sort(x)  # Sort ascending
    n = xs.size
    cumx = np.cumsum(xs)
    # Standard Gini formula for nonnegative data
    return (n + 1 - 2 * (cumx.sum() / cumx[-1])) / n


def _participation_ratio(var):
    # Measures how spread the variance is across coordinates
    s1 = var.sum()
    s2 = (var**2).sum()
    return (s1**2) / (len(var) * s2) if s1 > 0 and s2 > 0 else 0.0


var = Xc.var(axis=0, ddof=1)  # Sample variance per coordinate on centered data
var_sorted = np.sort(var)[::-1]  # Sort descending to see the heavy tail
gini = _gini_nonneg(var_sorted)
pr = _participation_ratio(var_sorted)

plt.figure(figsize=(7, 4))
plt.plot(np.arange(1, len(var_sorted) + 1), var_sorted, linewidth=2)  # Variance curve
plt.yticks([0.0000, 0.0001, 0.0002, 0.0003, 0.0004, 0.0005, 0.0006, 0.0007, 0.0008])
plt.xticks([0, 500, 1000, 1500, 2000, 2500, 3000, 3500, 4000])
plt.xlabel("Dimensions sorted by variance")
plt.ylabel("Variance")
plt.title(
    f"Qwen3-Embedding-8B"
)  # Report the two scalar summaries
plt.tight_layout()
plt.savefig(os.path.join(OUT, "anisotropy_variance_curve.png"), dpi=200)
plt.close()
print("Gini: " + str(gini))

# 2D projection: color by language, marker by type, with % variance in axis labels
Z2 = Z[:, :2]  # First two PC scores for each document
lang_colors = {
    "english": "tab:blue",
    "finnish": "tab:green",
    "swedish": "tab:red",
}  # Consistent colors
type_markers = {
    "original": "o",
    "delete": "x",
    "modify": "s",
    "paraphrase": "^",
}  # Markers per edit type

# Tiny deterministic jitter to reduce point overlap in the scatter (visual clarity)
rng = np.random.default_rng(42)  # Fixed seed for reproducibility
std_xy = Z2.std(axis=0, ddof=1)
jitter = rng.normal(
    scale=0.01 * std_xy, size=Z2.shape
)  # 1 percent of spread in each axis
Z2j = Z2 + jitter

plt.figure(figsize=(7, 6))
ax = plt.gca()
for (
    lang,
    color,
) in lang_colors.items():  # Loop language first for consistent legend colors
    for typ, marker in type_markers.items():  # Then loop type for different markers
        m = (meta["language"] == lang) & (meta["type"] == typ)
        if not m.any():
            continue
        if marker == "x":  
            ax.scatter(
                Z2j[m, 0],
                Z2j[m, 1],
                c=color,
                marker=marker,
                s=30,
                linewidths=1.0,
                alpha=0.85,
            )
        else:
            ax.scatter(
                Z2j[m, 0],
                Z2j[m, 1],
                c=color,
                marker=marker,
                s=30,
                alpha=0.85,
                linewidths=0.3,
            )

ax.set_xlabel(f"PC1 ({evr[0] * 100:.1f}%)")  # Percent variance explained on axes
ax.set_ylabel(f"PC2 ({evr[1] * 100:.1f}%)")
ax.set_title("PCA 2D projection")

# Separate legends (Language top-right, Type bottom-right)
lang_handles = [
    Line2D(
        [0],
        [0],
        marker="o",
        linestyle="None",
        markerfacecolor=c,
        markeredgecolor=c,
        label=lang,
    )
    for lang, c in lang_colors.items()
]
type_handles = [
    Line2D([0], [0], marker=m, linestyle="None", color="k", label=t)
    for t, m in type_markers.items()
]
leg_lang = ax.legend(
    handles=lang_handles, title="Language", loc="upper left", bbox_to_anchor=(1.02, 1)
)  
ax.add_artist(leg_lang)
ax.legend(
    handles=type_handles, title="Type", loc="lower left", bbox_to_anchor=(1.02, 0)
)

plt.tight_layout()
plt.savefig(os.path.join(OUT, "pca_2d.png"), dpi=200)  
plt.close()

# 2D trajectories: crisp quiver arrows from each original to each perturbation of the same (language, case)
plt.figure(figsize=(7, 6))
ax = plt.gca()
ax.set_xlabel(f"PC1 ({evr[0] * 100:.1f}%)")
ax.set_ylabel(f"PC2 ({evr[1] * 100:.1f}%)")
ax.set_title("Original → perturbation trajectories (PC1–PC2)")

# Originals as hollow circles, colored by language (easier to see arrow starts)
for lang, color in lang_colors.items():
    m = (meta["language"] == lang) & (meta["type"] == "original")
    ax.scatter(
        Z2[m, 0],
        Z2[m, 1],
        facecolors="none",
        edgecolors=color,
        s=80,
        linewidths=1.5,
        zorder=4,
    )

# Map (language, case_id) to original index so we know each arrow's start point
orig_idx = {
    (row["language"], row["case_id"]): i
    for i, row in meta[meta["type"] == "original"].iterrows()
}

# Draw arrows for each perturbation type in a different color
arrow_colors = {"paraphrase": "tab:orange", "modify": "tab:purple", "delete": "tab:brown"}
for typ, col in arrow_colors.items():
    xs, ys, us, vs = [], [], [], []  # Start points and vectors
    for i, row in meta[meta["type"] == typ].iterrows():
        j = orig_idx.get((row["language"], row["case_id"]))
        if j is None:
            continue
        x0, y0 = Z2[j, 0], Z2[j, 1]  # Original (start)
        x1, y1 = Z2[i, 0], Z2[i, 1]  # Perturbed (end)
        xs.append(x0)
        ys.append(y0)
        us.append(x1 - x0)
        vs.append(y1 - y0)  # Arrow components
    ax.quiver(  # Batch draw for cleaner arrows
        xs,
        ys,
        us,
        vs,
        angles="xy",
        scale_units="xy",
        scale=1,
        color=col,
        alpha=0.85,
        width=0.0035,
        headwidth=5.5,
        headlength=6.5,
        zorder=3,
    )

# Legends arranged like the PCA scatter: Language top-right, Type bottom-right
orig_handles = [
    Line2D(
        [0],
        [0],
        marker="o",
        linestyle="None",
        markerfacecolor="none",
        markeredgecolor=c,
        markersize=8,
        label=lang,
    )
    for lang, c in lang_colors.items()
]
type_arrow_handles = [
    Line2D([0], [0], color=c, lw=2.5, label=t) for t, c in arrow_colors.items()
]

leg_lang = ax.legend(
    handles=orig_handles,
    title="Language",
    loc="upper left",
    bbox_to_anchor=(1.02, 1),
    frameon=True,
    framealpha=0.95,
)
ax.add_artist(leg_lang)
ax.legend(
    handles=type_arrow_handles,
    title="Type",
    loc="lower left",
    bbox_to_anchor=(1.02, 0),
    frameon=True,
    framealpha=0.95,
)

plt.tight_layout()
plt.savefig(
    os.path.join(OUT, "pca_2d_trajectories_quiver.png"), dpi=200
)  
plt.close()

# Save fitted model and scores for later reuse
with open(os.path.join(OUT, "pca_model.pkl"), "wb") as f:
    pickle.dump({"pca": pca_doc}, f)
np.save(os.path.join(OUT, "pca_scores_2d.npy"), Z2)

print("Saved figures in " + OUT_DIR + "/pca_simple")

Gini: 0.2937243581504734
Saved figures in PCA_qwen3/pca_simple


### 4. k-Nearest Neighbors (kNN)

In [17]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.decomposition import PCA
from sklearn.preprocessing import normalize
from sklearn.neighbors import KNeighborsClassifier, NearestNeighbors

# Pairing each perturbation with its original inside the same (language, case_id),
# and a binary label: 1=meaningful (delete/modify), 0=paraphrase

def build_edit_vectors_once(meta: pd.DataFrame, X: np.ndarray):
    """
    Pairs each edited note with its original within (language, case_id).
    Label: 1 = meaningful (delete or modify), 0 = paraphrase.
    Returns:
      pairs DataFrame, V edit vectors, y labels, cases array for CV
    """
    orig_lookup = (
        meta.query("type == 'original'")
        .reset_index()
        .set_index(["language", "case_id"])["index"]
        .to_dict()
    )
    rows = []
    for i, r in meta.iterrows():
        if r["type"] in ("delete", "modify", "paraphrase"):
            j = orig_lookup.get((r["language"], r["case_id"]))
            if j is None:
                continue
            label = 1 if r["type"] in ("delete", "modify") else 0
            rows.append((i, j, str(r["case_id"]), label))
    pairs = pd.DataFrame(rows, columns=["i", "j", "case", "label"])

    # Cosine geometry via L2-normalization, then Euclidean difference
    Xn = normalize(X, norm="l2", axis=1, copy=True)
    V = Xn[pairs["i"].values] - Xn[pairs["j"].values]
    y = pairs["label"].astype(int).values
    cases = pairs["case"].values
    return pairs, V, y, cases

pairs, V, y, cases = build_edit_vectors_once(meta, X)


In [18]:
# SETUP

PC_LIST = [None, 2, 4, 8, 12, 16, 24, 32]  # None = Raw edits, no PCA
K = 3
W = "distance"


# Validation accuracy by space (leave-one-case-out)

def eval_cv_val_only(V, y, cases):
    rows = []
    unique_cases = np.unique(cases)
    for n_comp in PC_LIST:
        accs, tag = [], "raw_edits" if n_comp is None else f"pca@{n_comp}"
        for hold in unique_cases:
            tr, te = cases != hold, cases == hold
            Xtr, Xte, ytr, yte = V[tr], V[te], y[tr], y[te]

            if n_comp is None:
                Ztr, Zte = Xtr, Xte
            else:
                n_eff = min(n_comp, Xtr.shape[1], Xtr.shape[0])
                pca = PCA(n_components=n_eff, random_state=0)
                Ztr = pca.fit_transform(Xtr)
                Zte = pca.transform(Xte)

            if len(Zte):
                knn = KNeighborsClassifier(
                    n_neighbors=min(K, len(Ztr)), weights=W, metric="euclidean"
                )
                knn.fit(Ztr, ytr)
                accs.append(float(np.mean(knn.predict(Zte) == yte)))

        rows.append(
            {"space": tag, "mean_acc": np.nanmean(accs), "std_acc": np.nanstd(accs)}
        )
    order = [
        "raw_edits",
        "pca@2",
        "pca@4",
        "pca@8",
        "pca@12",
        "pca@16",
        "pca@24",
        "pca@32",
    ]
    return pd.DataFrame(rows).set_index("space").reindex(order)


# Leave-one-out training accuracy inside each fold

def loo_train_accuracy(Ztr, ytr, k=3, weights="distance"):
    n = len(Ztr)
    if n == 0:
        return np.nan
    k_eff = min(k + 1, n)  # include self then drop it
    nbrs = NearestNeighbors(n_neighbors=k_eff, metric="euclidean").fit(Ztr)
    dists, inds = nbrs.kneighbors(Ztr, return_distance=True)
    # Drop the self neighbor (first column)
    dists = dists[:, 1:]
    inds = inds[:, 1:]

    if weights == "distance":
        w = 1.0 / np.clip(dists, 1e-12, None)
    else:
        w = np.ones_like(dists)

    preds = np.empty(n, dtype=ytr.dtype)
    for i in range(n):
        neigh_labels = ytr[inds[i]]
        w0 = w[i][neigh_labels == 0].sum()
        w1 = w[i][neigh_labels == 1].sum()
        preds[i] = 1 if w1 >= w0 else 0
    return float((preds == ytr).mean())


def eval_cv_train_val(V, y, cases):
    rows = []
    unique_cases = np.unique(cases)
    for n_comp in PC_LIST:
        tag = "raw_edits" if n_comp is None else f"pca@{n_comp}"
        val_accs, train_accs = [], []
        for hold in unique_cases:
            tr, te = cases != hold, cases == hold
            Xtr, Xte, ytr, yte = V[tr], V[te], y[tr], y[te]

            if n_comp is None:
                Ztr, Zte = Xtr, Xte
            else:
                n_eff = min(n_comp, Xtr.shape[1], Xtr.shape[0])
                pca = PCA(n_components=n_eff, random_state=0)
                Ztr = pca.fit_transform(Xtr)
                Zte = pca.transform(Xte)

            # validation
            if len(Zte):
                knn = KNeighborsClassifier(
                    n_neighbors=min(K, len(Ztr)), weights=W, metric="euclidean"
                )
                knn.fit(Ztr, ytr)
                val_accs.append(float(np.mean(knn.predict(Zte) == yte)))

            # leave-one-out train accuracy
            train_accs.append(loo_train_accuracy(Ztr, ytr, k=K, weights=W))

        rows.append(
            {
                "space": tag,
                "val_mean": np.nanmean(val_accs),
                "val_std": np.nanstd(val_accs),
                "train_mean": np.nanmean(train_accs),
                "train_std": np.nanstd(train_accs),
            }
        )
    order = [
        "raw_edits",
        "pca@2",
        "pca@4",
        "pca@8",
        "pca@12",
        "pca@16",
        "pca@24",
        "pca@32",
    ]
    return pd.DataFrame(rows).set_index("space").reindex(order)


# RUN, PRINT AND PLOT

res = eval_cv_val_only(V, y, cases)
tv = eval_cv_train_val(V, y, cases)

# One-liner for Results, k-Nearest Neighbours
train_m = float(tv.loc["raw_edits", "train_mean"])
val_m = float(tv.loc["raw_edits", "val_mean"])
print(
    f"Mean training accuracy was {train_m:.2f} across folds, "
    f"and validation {val_m:.2f}" 
)

# Pretty table for the paper
print("\nTrain and validation across spaces:")
print(tv[["val_mean", "val_std", "train_mean", "train_std"]].round(3))

# Figure for paper, Δ vs raw plot
raw_m = float(res.loc["raw_edits", "mean_acc"])
raw_s = float(res.loc["raw_edits", "std_acc"])
labels = ["pca@2", "pca@4", "pca@8", "pca@12", "pca@16", "pca@24", "pca@32"]
means = res.loc[labels, "mean_acc"].values.astype(float)
stds = res.loc[labels, "std_acc"].values.astype(float)

deltas = means - raw_m
errs = np.sqrt(stds**2 + raw_s**2)

fig, ax = plt.subplots(figsize=(7, 3.6), dpi=130)
ax.axhline(0.0, linestyle="--")
ax.errorbar(np.arange(len(labels)), deltas, yerr=errs, fmt="o-")
for x, d in enumerate(deltas):
    ax.text(
        x,
        d + (0.02 if d >= 0 else -0.02),
        f"{d:+.3f}",
        ha="center",
        va="bottom" if d >= 0 else "top",
        fontsize=9,
    )
ax.set_xticks(range(len(labels)))
ax.set_xticklabels(labels)
ax.set_ylabel("Δ accuracy vs raw (meaningful vs paraphrase)")
ax.set_title("PCA on edit vectors for meaningful-edit detection")
plt.tight_layout()

# Save figure
out_dir = os.environ.get("OUT_DIR", ".")
plot_path = os.path.join(OUT_DIR, "meaningful_edits_delta_vs_raw.png")
plt.savefig(plot_path, bbox_inches="tight")
plt.close()
print("Saved plot:", plot_path)


Mean training accuracy was 0.94 across folds, and validation 0.38

Train and validation across spaces:
           val_mean  val_std  train_mean  train_std
space                                              
raw_edits     0.378    0.072       0.939      0.015
pca@2         0.489    0.086       0.824      0.098
pca@4         0.585    0.092       0.924      0.020
pca@8         0.400    0.072       0.974      0.012
pca@12        0.400    0.059       0.967      0.009
pca@16        0.393    0.083       0.948      0.014
pca@24        0.378    0.054       0.941      0.017
pca@32        0.378    0.059       0.939      0.017
Saved plot: PCA_qwen3\meaningful_edits_delta_vs_raw.png
